# 02b — Modelo Temporal (ResNet50V2 + BiGRU + Attention)

Extensión del enfoque frame-a-frame: en vez de clasificar frames aislados,
clasificamos **secuencias de 16 frames consecutivos** (~6.4 s a 2.5 fps) usando
el backbone ResNet50V2 entrenado + una cabeza temporal BiGRU con Attention.

**Motivación:** la somnolencia es un fenómeno temporal — un frame aislado
no la captura. El modelo frame-a-frame alcanza un techo de ~0.68 F1 por
atajos de sesión y labels ruidosas. El modelo temporal agrega contexto temporal.

> **Alternativa si el kernel falla en WSL2:**
> ```bash
> source /home/lilidl/pnl/.venv/bin/activate
> cd /home/lilidl/tp_final_cv2
> python train_temporal.py
> ```
> Checkpoints y curvas se guardan en `checkpoints/` y `docs/`.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch

from src.data.dataset import make_splits
from src.data.sequence_dataset import make_sequence_loaders
from src.models.temporal_model import TemporalDrowsinessModel
from src.training.train_temporal import train_temporal

## 1. Configuración

In [ ]:
H5            = '/home/lilidl/drowsiness_crops.h5'
BACKBONE_CKPT = Path('../checkpoints/resnet50v2_best.pt')
KEEP_CLASSES  = [0, 2]
LABEL_MAP     = {0: 0, 2: 1}
SEQ_LEN       = 16     # ~6.4 s a 2.5 fps
TRAIN_STRIDE  = 4
EVAL_STRIDE   = 16
BATCH_SIZE    = 32
NUM_WORKERS   = 8

## 2. Splits y sequence loaders

In [ ]:
train_idx, val_idx, test_idx = make_splits(H5, keep_classes=KEEP_CLASSES)

train_loader, val_loader, test_loader = make_sequence_loaders(
    H5, train_idx, val_idx, test_idx,
    seq_len=SEQ_LEN, train_stride=TRAIN_STRIDE, eval_stride=EVAL_STRIDE,
    batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, label_map=LABEL_MAP,
)
print(f'train secuencias: {len(train_loader.dataset):,}')
print(f'val   secuencias: {len(val_loader.dataset):,}')
print(f'test  secuencias: {len(test_loader.dataset):,}')

## 3. Entrenar modelo temporal

In [ ]:
CHECKPOINT = Path('../checkpoints/temporal_best.pt')
HISTORY    = Path('../checkpoints/history_temporal.json')

if HISTORY.exists():
    with open(HISTORY) as f:
        history = json.load(f)
    print(f'Cargado desde {HISTORY}')
    print(f'Mejor val F1: {max(history["val_f1"]):.4f}')
else:
    model = TemporalDrowsinessModel(
        backbone_checkpoint=BACKBONE_CKPT,
        seq_len=SEQ_LEN, hidden=512, layers=2, dropout=0.3, num_classes=2,
    )
    print(f'Params totales:     {model.count_total():,}')
    print(f'Entrenables fase 1: {model.count_trainable():,}')
    history = train_temporal(
        model=model, train_loader=train_loader, val_loader=val_loader,
        checkpoint_path=CHECKPOINT,
        epochs_frozen=10, epochs_unfrozen=25,
        lr_head=1e-3, lr_finetune=5e-5, patience=8, unfreeze_blocks=2,
    )
    with open(HISTORY, 'w') as f:
        json.dump(history, f, indent=2)

## 4. Curvas de entrenamiento

In [ ]:
fig, (ax_loss, ax_f1) = plt.subplots(1, 2, figsize=(14, 5))
epochs = range(1, len(history['train_loss']) + 1)
phase_change = next(i for i in range(1, len(history['phase']))
                    if history['phase'][i] != history['phase'][i-1])

for ax, key, title in [(ax_loss, 'loss', 'Loss'), (ax_f1, 'f1', 'F1-macro')]:
    ax.plot(epochs, history[f'train_{key}'], 'b--', alpha=0.6, label='train')
    ax.plot(epochs, history[f'val_{key}'],   'b-',              label='val')
    ax.axvline(phase_change + 0.5, color='gray', linestyle=':', alpha=0.5)
    ax.set_xlabel('Epoch'); ax.set_ylabel(title)
    ax.set_title(f'Temporal — {title}'); ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../docs/curvas_temporal.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Mejor val F1-macro: {max(history["val_f1"]):.4f}')